# 13. Training Dynamics — Model M（BPE 8K, pilot 10M, <1 epoch）

## 学習目標
- 実コーパス・BPE・1エポック未満という M1 と異なる条件での学習曲線を読む
- train と validation が**別文書**のときのギャップの意味（汎化）を M1（同一小説）と対比する
- 横軸を step / tokens / 時間で切り替えて解釈する

## 前提
BPE 語彙 8192 なので一様分布損失は $\ln 8192 = 9.01$ nats。M1 の char（$\ln 2068=7.63$）とは単位が違うので、損失値の直接比較はできない（perplexity も同様）。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
from jp_llm_lab.visualization import curves
records = read_jsonl(M2_RUN / "metrics.jsonl")
config = load_json(M2_RUN / "config.json")
summary = load_json(M2_RUN / "summary.json")
print(f'params {config["param_breakdown"]["total"]:,}  '
      f'tokens {summary["tokens_seen"]:,}  wallclock {summary["wallclock_sec"]:.1f}s')
curves.loss_curves_figure(records).show()

params 10,183,680  tokens 9,830,400  wallclock 40.1s


In [3]:
evals = [r for r in records if r["type"]=="eval"]
ln_v = math.log(8192)
print(f"init val loss {evals[0]['val_eval']['loss']:.2f}  (uniform ln V = {ln_v:.2f})")
print(f"final train {evals[-1]['train_eval']['loss']:.2f}  val {evals[-1]['val_eval']['loss']:.2f}"
      f"  gap {evals[-1]['train_eval']['loss']-evals[-1]['val_eval']['loss']:+.3f}")
print(f"final val ppl {evals[-1]['val_eval']['ppl']:.0f}  top1_conf {evals[-1]['val_eval']['top1_conf']:.3f}")

init val loss 9.07  (uniform ln V = 9.01)
final train 6.04  val 6.05  gap -0.009
final val ppl 423  top1_conf 0.064


In [4]:
curves.lr_grad_figure(records, config["train_config"]["grad_clip"]).show()
curves.tokens_per_sec_figure(records).show()

## Observation / Interpretation / Caveat
- **Observation**: init loss ≈ ln V。train と val がほぼ重なったまま低下し、M1 で見た train≪val の乖離が**ない**。1エポック未満なので同じ文書を繰り返し見ていない。
- **Interpretation**: train/val ギャップが小さい＝暗記していない＝損失改善は（この範囲では）汎化由来。これが M1 の「45エポックで暗記」との決定的な違い。ただし val loss 6.0 (ppl 400+) は絶対的には高く、10M params × 10M tokens では web テキストのモデル化は浅い。
- **Caveat**: BPE の loss/ppl は char と比較不可（単位が違う）。val は FineWeb2 の別文書で、話題ドリフトを含む。1シードのみ（M3 で複数シード）。

**次へ**: [14_attention_visualization](14_attention_visualization.ipynb)